# ASR demo

Вставьте публичную ссылку на ZIP- или TAR-архив с вашим датасетом и выберите Runtime > Run all. В корне архива должна быть папка `audio/` и, при наличии эталонов, `transcriptions/`. Имена аудио и текстов должны совпадать.



In [ ]:
DATASET_URL = ""  # string


## Установка

Используем Python текущего notebook и устанавливаем зависимости через `pip`. 


In [ ]:
import shutil
from pathlib import Path

%cd /content
PROJECT = Path.cwd() / "ASR"
if not PROJECT.exists():
    !git clone --branch example/asr --single-branch https://github.com/romaagarev/ASR.git "{PROJECT}"

%cd ASR
%pip install -q -r requirements_demo.txt
%pip install -q --no-deps -r requirements_lm.txt

## Веса

Скачиваем чекпоинт модели. Из тренировочного чекпоинта оставляем только веса для инференса.


In [ ]:
CHECKPOINT_URL = "https://drive.google.com/file/d/1t9gCYH3H8kbbj-LSVBvOecSrBngrMs3W/view?usp=share_link"
CHECKPOINT = PROJECT / "saved/demo/model.pth"


In [ ]:
%run download.py "{CHECKPOINT}" --url "{CHECKPOINT_URL}"


In [ ]:
import torch

checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
torch.save(checkpoint.get("state_dict", checkpoint), CHECKPOINT)


## Запуск инференса и метрик

Используем GRU learn4 и [ByGPT5-small-en](https://huggingface.co/nllg/bygpt5-small-en): beam 64, вес LM 0.7, бонус за слова 0.5. Параметры заданы в `inference_deepspeech2.yaml`.

Предсказания сохраняются отдельными текстовыми файлами.


In [ ]:
def recognize(dataset_dir, output_dir):
    shutil.rmtree(output_dir, ignore_errors=True)
    %run inference.py --config-name inference_deepspeech2 \
        "input_dir={dataset_dir}" \
        "inferencer.from_pretrained={CHECKPOINT}" \
        "inferencer.save_path={output_dir}"
    predictions = output_dir / "custom"
    targets = dataset_dir / "transcriptions"
    if any(targets.glob("*.txt")):
        %run calc_metrics.py "{targets}" "{predictions}"
    return predictions


## Пример на LibriSpeech

Три записи из публичного набора `librispeech_asr_dummy`(https://huggingface.co/datasets/hf-internal-testing/librispeech_asr_dummy)


In [ ]:
EXAMPLE_DIR = PROJECT / "data/demo/example"
%run download.py "{EXAMPLE_DIR}" --example
RESULT_DIR = recognize(EXAMPLE_DIR, PROJECT / "data/demo/predictions_example")

from IPython.display import Audio, display

for audio in sorted((EXAMPLE_DIR / "audio").glob("*.flac")):
    display(Audio(filename=str(audio)))
    print("Target:", (EXAMPLE_DIR / "transcriptions" / f"{audio.stem}.txt").read_text())
    print("Prediction:", (RESULT_DIR / f"{audio.stem}.txt").read_text())


## Ваш датасет

Скачиваем архив по `DATASET_URL` и распаковываем его. В корне архива ожидаются `audio/` и необязательная `transcriptions/`. При отсутствии эталонов будут сохранены только предсказания


In [ ]:
if DATASET_URL:
    DATA_DIR = PROJECT / "data/demo/user_dataset"
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    %run download.py "{DATA_DIR}" --url "{DATASET_URL}" --unpack
    RESULT_DIR = recognize(DATA_DIR, PROJECT / "data/demo/predictions_user")
    audio_files = {path.stem: path for path in (DATA_DIR / "audio").iterdir()}
    for prediction in sorted(RESULT_DIR.glob("*.txt"))[:3]:
        display(Audio(filename=str(audio_files[prediction.stem])))
        target = DATA_DIR / "transcriptions" / prediction.name
        if target.is_file():
            print("Target:", target.read_text())
        print("Prediction:", prediction.read_text())
